In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin

warnings.filterwarnings('ignore')

# ==========================================
# [0] 데이터 로드 및 병합
# ==========================================
print("1. 데이터 불러오는 중...")
triage   = pd.read_csv("triage.csv")
patients = pd.read_csv("patients.csv")

# 환자 기본 정보 병합 및 성별 인코딩
df = triage.merge(
    patients[['subject_id', 'anchor_age', 'gender']],
    on='subject_id', how='left'
)
df['gender'] = df['gender'].map({'M': 1, 'F': 0})

# 타겟(acuity) 결측치는 훈련에 방해되므로 미리 제거
df.dropna(subset=['acuity'], inplace=True)
df = df.reset_index(drop=True)

print(f"초기 병합 데이터 크기: {df.shape}")

# ==========================================
# ★ [0.5단계] 주증상(CC) 대분류 카테고리화 (Train/Test 분리 전)
# ==========================================
print("2. 주증상(CC) 카테고리화 진행 중...")

# 1. 텍스트 결측치 처리 및 전처리
df['chiefcomplaint'] = df['chiefcomplaint'].fillna('').astype(str).str.upper()

exact_replacements = {
    'HA': 'H/A', 'AH': 'A/H', 'OD': 'OVERDOSE', 
    'SI': 'SUICIDAL', 'ST': 'TACHYCARDIA'
}
df['chiefcomplaint'] = df['chiefcomplaint'].replace(exact_replacements)

# 2. R 코드 기반 소분류 키워드 딕셔너리
small_dict = {
    "abd_pain": ["ABDOMINAL PAIN", "ABDO PAIN", "ABD PAIN", "AB PAIN", "EPIGASTRIC PAIN", "ABDOMINAL DISTENTION", "PELVIC PAIN", "RIB PAIN", "RLQ PAIN", "RUQ PAIN"],
    "chest_pain": ["CHEST PAIN", "CP", "C/P"],
    "transfer": ["TRANSFER"],
    "fall": ["FALL"],
    "dyspnea": ["DYSPNEA", "SOB", "SHORTNESS OF BREATH", "SHORT OF BREATH", "WHEEZING", "DIFFICULTY BREATHING", "RESPIRATORY DISTRESS"],
    "headache": ["HEADACHE", "H/A", "HEAD ACHE", "MIGRAINE"],
    "fever": ["FEVER"],
    "intox": ["ETOH", "SUBSTANCE", "DETOX", "WITHDRAWAL"],
    "overdose": ["OVERDOSE"],
    "nvd": ["N/V", "NAUSEA", "DIARRHEA", "VOMITING"],
    "back_pain": ["BACK", "LBP"],
    "weak_and_dizzy": ["DIZZINESS", "WEAKNESS", "LIGHTHEADED", "SYNCOPE", "PRESYNCOPE", "FATIGUE", "LETHARGY", "DIZZY", "VERTIGO"],
    "wound": ["WOUND", "LACERATION"],
    "suicidal": ["SUICIDAL", "SELF HARM", "ATTEMPT"],
    "cough": ["COUGH"],
    "abnormal_labs": ["ABNORMAL", "ABNL"],
    "ili": ["ILI", "INFLUENZA", "FLU SYMPTOMS", "FLU LIKE", "FLU-LIKE"],
    "confusion": ["ALTERED MENTAL STATUS", "CONFUSION", "AGITATION"],
    "mental_illness": ["ANXIETY", "DEPRESSION", "PSYCH", "HALLUCINATIONS", "A/H", "DELUSION", "MANIA"],
    "mvc": ["MVC", "PED STRUCK", "BICYCLE ACCIDENT", "MOTORCYCLE ACCIDENT"],
    "head_injury": ["HEAD INJURY", "HEAD BLEED", "HEADBLEED", "HEAD LAC"],
    "sore_throat": ["SORE THROAT"],
    "palpitations": ["PALPITATIONS", "TACHYCARDIA", "ABNORMAL EKG", "BRADYCARDIA", "FIBRILLATION"],
    "rectal_pain": ["RECTAL PAIN"],
    "rectal_bleeding": ["BRBPR", "RECTAL BLEEDING", "BLEEDING FROM RECTUM", "BLOOD IN STOOL", "BLOODY STOOL", "MELENA", "HEMATURIA", "BLOOD IN URINE", "HEMATEMESIS", "HEMOPTYSIS", "COFFEE GROUND EMESIS"],
    "neck_pain": ["NECK"],
    "seizures": ["SEIZURE"],
    "rash": ["RASH"],
    "dysuria": ["DYSURIA", "URINARY RETENTION", "URINARY FREQUENCY"],
    "flank_pain": ["FLANK"],
    "walking_problems": ["UNSTEADY GAIT", "UNABLE TO AMBULATE", "DIFFICULTY AMBULATING", "INABILITY TO AMBULATE", "DIFF AMBULATING", "CAN'T WALK", "CANT WALK", "UNABLE TO WALK", "VAGINAL SPOTTING"],
    "hypertension": ["HYPERTENSION", "HIGH BLOOD PRESSURE"],
    "vaginal_bleeding": ["VAGINAL BLEEDING", "VAG BLEEDING", "VAG BLEED", "VAGINAL BLEED", "VAGINAL PAIN", "VAGINAL SPOTTING",  "VAG SPOTTING"],
    "blood_sugar": ["HYPERGLYCEMIA", "HYPOGLYCEMIA", "BLOOD SUGAR"],
    "allergy": ["ALLERGIC", "ALLERGY"],
    "pregnant": ["PREG", "LABOR"],
    "knee_pain": ["KNEE"],
    "assault": ["ASSAULT"],
    "low_blood_pressure": ["HYPOTENSION", "LOW BLOOD PRESSURE"],
    "foot_pain": ["FOOT", "ANKLE", "TOE"],
    "vision_issues": ["VISION", "SEEING FLOATERS", "BLIND", "VISUAL"],
    "arm_pain": ["ARM", "ELBOW"],
    "shoulder_pain": ["SHOULDER"],
    "hip_pain": ["HIP"],
    "abscess": ["ABCESS"],
    "asthma": ["ASTHMA"],
    "nosebleed": ["EPISTAXIS"],
    "hypoxia": ["HYPOXIA"],
    "hematoma": ["ICH", "SDH", "SAH"],
    "leg_pain_or_swelling": ["LEG", "CALF"],
    "eye_pain": ["EYE PAIN"],
    "ear_pain": ["EAR PAIN"],
    "numbness": ["NUMBNESS"],
    "hand_finger_wrist": ["HAND", "FINGER", "WRIST"]
}

# 3. 엑셀 이미지에서 추출한 12개 대분류 매핑
large_category_mapping = {
    "Gastrointestinal": ["abd_pain", "nvd", "rectal_bleeding", "rectal_pain"],
    "Cardiovascular": ["chest_pain", "palpitations", "low_blood_pressure", "hypertension"],
    "Respiratory": ["dyspnea", "cough", "hypoxia", "asthma"],
    "Neurological": ["headache", "weak_and_dizzy", "confusion", "seizures", "numbness", "hematoma"],
    "Trauma_Injury": ["fall", "wound", "mvc", "head_injury", "assault"],
    "Psychiatric_Substance": ["intox", "overdose", "suicidal", "mental_illness"],
    "Infection_Immune": ["fever", "ili", "allergy", "abscess"],
    "Musculoskeletal_Pain": ["back_pain", "neck_pain", "flank_pain", "knee_pain", "foot_pain", "arm_pain", "shoulder_pain", "hip_pain", "leg_pain_or_swelling", "hand_finger_wrist"],
    "Genitourinary_Obstetric": ["dysuria", "vaginal_bleeding", "pregnant"],
    "ENT_Ophthalmology": ["sore_throat", "vision_issues", "nosebleed", "eye_pain", "ear_pain"],
    "Endocrine_Metabolic": ["blood_sugar"],
    "General_Other": ["transfer", "abnormal_labs", "walking_problems", "rash"]
}

cc_large_cols = list(large_category_mapping.keys())

# 4. 환자 데이터에 12개 대분류 One-Hot Encoding 적용
for large_cat, small_cats in large_category_mapping.items():
    df[large_cat] = False 
    for small_cat in small_cats:
        if small_cat not in small_dict: continue 
        for s in small_dict[small_cat]:
            df[large_cat] = df[large_cat] | df['chiefcomplaint'].str.contains(s, regex=False)
            
    df[large_cat] = df[large_cat].astype(int)

# 5. 주증상 원본 컬럼 삭제
df.drop(columns=['chiefcomplaint'], inplace=True)

# ==========================================
# [1단계] 이상치 → NaN / 과다 결측 제거
# ==========================================
vitals = ['sbp', 'dbp', 'heartrate', 'resprate', 'temperature', 'o2sat']
bounds = {
    'sbp':         (50,  260),
    'dbp':         (20,  160),
    'heartrate':   (25,  225),
    'resprate':    (7,   40),
    'temperature': (86,  113),
    'o2sat':       (50,  100),
}

for col, (lo, hi) in bounds.items():
    if col in df.columns:
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

df['missing_count'] = df[vitals].isnull().sum(axis=1)
df = df[df['missing_count'] < 4].drop(columns=['missing_count'])
df = df.reset_index(drop=True)

# ==========================================
# [2단계] 결측 지시자 생성
# ==========================================
for col in vitals:
    df[f'{col}_missing'] = df[col].isnull().astype(int)

missing_cols = [f'{col}_missing' for col in vitals]
impute_cols  = vitals + ['anchor_age', 'gender']

# ==========================================
# [3단계] Train / Test Split
# ==========================================
print("3. 데이터를 Train/Test로 분리합니다...")
# ★ 대분류 컬럼(cc_large_cols) 추가 완료 ★
feature_cols = impute_cols + missing_cols + cc_large_cols
X = df[feature_cols].copy()
y = df['acuity'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train 셋: {X_train.shape}, Test 셋: {X_test.shape}")

# ==========================================
# [4단계] MICE 대치기 커스텀 클래스
# ==========================================
class VitalImputer(BaseEstimator, TransformerMixin):
    def __init__(self, vitals, bounds, n_quantiles=1000, max_iter=10, random_state=42):
        self.vitals        = vitals
        self.bounds        = bounds
        self.n_quantiles   = n_quantiles
        self.max_iter      = max_iter
        self.random_state  = random_state
        self.impute_cols   = vitals + ['anchor_age', 'gender']

    def fit(self, X, y=None):
        df = X[self.impute_cols].copy() 

        # 1. 변수별 QT — Train 관측값으로만 학습
        self.qt_ = {}
        for col in self.vitals:
            obs = df[col].dropna().values.reshape(-1, 1)
            if len(obs) < 10:
                self.qt_[col] = None
                continue
            qt = QuantileTransformer(
                n_quantiles=min(len(obs), self.n_quantiles),
                output_distribution='normal',
                random_state=self.random_state
            )
            qt.fit(obs)
            self.qt_[col] = qt

        # 2. QT 적용하여 정규화
        df_t = self._qt_transform(df)

        # 3. MICE 학습
        self.imputer_ = IterativeImputer(
            estimator=BayesianRidge(),
            sample_posterior=True,
            max_iter=self.max_iter,
            random_state=self.random_state
        )
        self.imputer_.fit(df_t[self.impute_cols])
        return self

    def transform(self, X, y=None):
        df       = X.copy()
        df_vital = df[self.impute_cols].copy()
        
        # 1. QT 변환
        df_t     = self._qt_transform(df_vital)

        # 2. MICE 대치
        imputed  = self.imputer_.transform(df_t[self.impute_cols])
        df_t.loc[:, self.impute_cols] = imputed

        # 3. 역변환 전 극단값 1차 컷팅
        for col in self.vitals:
            if self.qt_.get(col) is not None:
                df_t.loc[:, col] = df_t[col].clip(-4, 4)
        df_inv = self._qt_inverse(df_t)

        # 4. 생물학적 한계치 2차 컷팅
        for col, (lo, hi) in self.bounds.items():
            if col in df_inv.columns:
                df_inv.loc[:, col] = df_inv[col].clip(lo, hi)

        # 5. 정수형 복원
        df_inv['gender']     = df_inv['gender'].round().clip(0,1).astype(int)
        df_inv['anchor_age'] = df_inv['anchor_age'].round().clip(0,120).astype(int)

        # 6. 최종 업데이트 (★ 원본 데이터 100% 보존, 결측치만 채움 ★)
        for col in self.impute_cols:
            df[col] = df[col].fillna(df_inv[col])
        return df

    def _qt_transform(self, df):
        result = df.copy()
        for col in self.vitals:
            qt = self.qt_.get(col)
            if qt is None: continue
            mask = result[col].notna()
            if mask.sum() > 0:
                result.loc[mask, col] = qt.transform(result.loc[mask, col].values.reshape(-1, 1)).ravel()
        return result

    def _qt_inverse(self, df):
        result = df.copy()
        for col in self.vitals:
            qt = self.qt_.get(col)
            if qt is None: continue
            mask = result[col].notna()
            if mask.sum() > 0:
                result.loc[mask, col] = qt.inverse_transform(result.loc[mask, col].values.reshape(-1, 1)).ravel()
        return result

# ==========================================
# [5단계] 대치 실행
# ==========================================
print("4. MICE 결측치 대치 진행 중 (시간이 조금 걸릴 수 있습니다)...")
imputer = VitalImputer(vitals=vitals, bounds=bounds)

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed  = imputer.transform(X_test)

# ==========================================
# [6단계] 전처리 모델 저장 및 완료
# ==========================================
joblib.dump(imputer, 'vital_imputer.pkl')
print("✅ 전처리 파이프라인 완료! (vital_imputer.pkl 저장됨)")

1. 데이터 불러오는 중...
초기 병합 데이터 크기: (418100, 13)
2. 주증상(CC) 카테고리화 진행 중...
3. 데이터를 Train/Test로 분리합니다...
Train 셋: (326442, 26), Test 셋: (81611, 26)
4. MICE 결측치 대치 진행 중 (시간이 조금 걸릴 수 있습니다)...
✅ 전처리 파이프라인 완료! (vital_imputer.pkl 저장됨)
